# Reconstruct annual and seasonal mass balance

## Notebook Guide

This notebook generates winter, summer, and annual WGMS reconstruction products using the global XGBoost models as the primary product. O1Region is retained for grouping, seasonal-window fallback, and regional aggregation, but no regional residual-bias correction is applied.

## 1. Winter reconstruction

In [ ]:
from pathlib import Path
import json
import math
import dill
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

from os import environ

configured_root = environ.get("ML4GM_ROOT")
ROOT_DIR = Path(configured_root).expanduser().resolve() if configured_root else Path.cwd().resolve()
if not configured_root:
    if (ROOT_DIR / "01_Dataset.ipynb").is_file():
        ROOT_DIR = ROOT_DIR.parent
    if not (ROOT_DIR / "code" / "01_Dataset.ipynb").is_file():
        raise RuntimeError("Start this notebook from the release root or its code directory, or set ML4GM_ROOT.")
elif not ROOT_DIR.is_dir():
    raise FileNotFoundError(f"The configured ML4GM_ROOT directory does not exist: {ROOT_DIR}")
WGMS_DIR = ROOT_DIR / "data/DOI-WGMS-FoG-2026-02-10/data"
TARGET_CSV = ROOT_DIR / "data/interim/wgms_targets/wgms_global_targets_with_rgi_1979_2025.csv"
ERA_LONG_CSV = ROOT_DIR / "data/raw/era5land_gee/wgms_global_era5land_1979_2025.csv"
OUT_DIR = ROOT_DIR / "data/processed/seasonal_training"
RECON_DIR = ROOT_DIR / "results/reconstruction"
MODEL_DIR = ROOT_DIR / "models/annual_seasonal"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RECON_DIR.mkdir(parents=True, exist_ok=True)

YEAR_MIN, YEAR_MAX = 1979, 2025
K_NEIGHBORS = 5
NN_VOTE_THRESHOLD = 0.5
REGION_ENSEMBLE_TOPK = 3
STATE_VARS = ['t2m', 'rsn', 'sde', 'tsn', 'fal', 'sp']
FLOW_VARS = ['sf', 'smlt', 'slhf', 'ssr', 'strd', 'sshf', 'ssrd', 'tp']
CLIMATE_VARS = STATE_VARS + FLOW_VARS
STATIC_FEATURES = ['Area', 'CenLon', 'CenLat', 'Zmin', 'Zmax', 'Zmed', 'Slope', 'Aspect', 'Lmax']


def parse_wgms_date(series):
    s = series.astype('string').str.strip()
    out = pd.to_datetime(pd.Series(pd.NaT, index=series.index))
    numeric_mask = s.str.fullmatch(r'\d{8}').fillna(False)
    out.loc[numeric_mask] = pd.to_datetime(s.loc[numeric_mask], format='%Y%m%d', errors='coerce')
    out.loc[~numeric_mask] = pd.to_datetime(s.loc[~numeric_mask], errors='coerce')
    return out


def shift_month(date_series, direction):
    adjusted = date_series.copy()
    if direction == 'start':
        adjusted = adjusted.where(adjusted.dt.day <= 15, adjusted + pd.offsets.MonthBegin(1))
    elif direction == 'end':
        adjusted = adjusted.where(adjusted.dt.day >= 15, adjusted - pd.offsets.MonthEnd(1))
    else:
        raise ValueError("direction must be 'start' or 'end'")
    return adjusted


def normalize_region(series):
    return pd.to_numeric(series, errors='coerce').astype('Int64')


def window_key(row):
    vals = [
        row['season_start_month'], row['season_end_month'],
        row['season_start_year_offset'], row['season_end_year_offset'],
    ]
    if any(pd.isna(v) for v in vals):
        return None
    return (int(vals[0]), int(vals[1]), int(vals[2]), int(vals[3]))


def window_pair_label(key):
    if key is None:
        return np.nan
    sm, em, so, eo = key
    return f"{so:+d}:{sm:02d}->{eo:+d}:{em:02d}"


def dominant_window_pair(group):
    keys = group.apply(window_key, axis=1).dropna()
    if keys.empty:
        return None, np.nan
    counts = keys.value_counts()
    top_count = counts.iloc[0]
    top_keys = sorted(counts[counts == top_count].index.tolist())
    key = top_keys[0]
    return key, float(top_count / counts.sum())


def apply_window_key(df, key, source, confidence, vote_share=np.nan, neighbor_count=0, neighbor_distance_median_km=np.nan, ensemble_size=1):
    sm, em, so, eo = key
    df = df.copy()
    df['season_start_month'] = sm
    df['season_end_month'] = em
    df['season_start_year_offset'] = so
    df['season_end_year_offset'] = eo
    df['window_pair'] = window_pair_label(key)
    df['season_window_source'] = source
    df['window_confidence'] = confidence
    df['window_vote_share'] = vote_share
    df['neighbor_count'] = neighbor_count
    df['neighbor_distance_median_km'] = neighbor_distance_median_km
    df['window_ensemble_size'] = ensemble_size
    return df


def month_sequence(start_month, start_offset, end_month, end_offset, balance_year):
    start_year = int(balance_year) + int(start_offset)
    end_year = int(balance_year) + int(end_offset)
    start_period = pd.Period(year=start_year, month=int(start_month), freq='M')
    end_period = pd.Period(year=end_year, month=int(end_month), freq='M')
    if end_period < start_period:
        return []
    periods = pd.period_range(start_period, end_period, freq='M')
    return [(int(p.year), int(p.month)) for p in periods]


def aggregate_month_window(records, era_lookup):
    rows = []
    for rec in records.itertuples(index=False):
        months = month_sequence(rec.season_start_month, rec.season_start_year_offset, rec.season_end_month, rec.season_end_year_offset, rec.year)
        values = []
        for year_month in months:
            key = (rec.RGIId, year_month[0], year_month[1])
            if key in era_lookup:
                values.append(era_lookup[key])
        if len(values) == 0:
            out = {var: np.nan for var in CLIMATE_VARS}
        else:
            frame = pd.DataFrame(values)
            out = {var: frame[var].mean() for var in STATE_VARS}
            out.update({var: frame[var].sum(min_count=1) for var in FLOW_VARS})
        rows.append(out)
    return pd.DataFrame(rows)


def prepare_base_tables():
    mass_balance = pd.read_csv(WGMS_DIR / 'mass_balance.csv', low_memory=False)
    target_meta = pd.read_csv(TARGET_CSV, low_memory=False)
    era_long = pd.read_csv(ERA_LONG_CSV, low_memory=False)

    target_meta['wgms_glacier_id'] = pd.to_numeric(target_meta['wgms_glacier_id'], errors='coerce').astype('Int64')
    target_meta['O1Region'] = normalize_region(target_meta['O1Region'])
    target_meta['O2Region'] = pd.to_numeric(target_meta['O2Region'], errors='coerce').astype('Int64')
    mapped_ids = target_meta['wgms_glacier_id'].dropna().unique().tolist()

    base_meta_cols = [
        'RGIId', 'GLIMSId', 'CenLon', 'CenLat', 'O1Region', 'O2Region', 'Area',
        'Zmin', 'Zmax', 'Zmed', 'Slope', 'Aspect', 'Lmax', 'Status', 'Connect',
        'Form', 'TermType', 'Surging', 'Linkages', 'Name', 'gtng_region',
        'glacier_name', 'wgms_glacier_id'
    ]
    meta_map = (
        target_meta[base_meta_cols]
        .dropna(subset=['wgms_glacier_id'])
        .sort_values(['RGIId'])
        .drop_duplicates(subset=['wgms_glacier_id'], keep='first')
        .reset_index(drop=True)
    )
    meta_by_rgi = (
        target_meta[base_meta_cols]
        .sort_values(['RGIId'])
        .drop_duplicates(subset=['RGIId'], keep='first')
        .reset_index(drop=True)
    )

    mb = mass_balance.copy()
    mb['glacier_id'] = pd.to_numeric(mb['glacier_id'], errors='coerce').astype('Int64')
    mb['year'] = pd.to_numeric(mb['year'], errors='coerce').astype('Int64')
    mb = mb.loc[mb['glacier_id'].isin(mapped_ids) & mb['year'].between(YEAR_MIN, YEAR_MAX)].copy()
    mb = mb.drop(columns=[c for c in ['glacier_name', 'gtng_region'] if c in mb.columns])
    mb = mb.merge(meta_map, left_on='glacier_id', right_on='wgms_glacier_id', how='inner')

    mb['begin_dt'] = parse_wgms_date(mb['begin_date'])
    mb['mid_dt'] = parse_wgms_date(mb['midseason_date'])
    mb['end_dt'] = parse_wgms_date(mb['end_date'])
    mb['O1Region'] = normalize_region(mb['O1Region'])
    mb['O2Region'] = pd.to_numeric(mb['O2Region'], errors='coerce').astype('Int64')

    usecols = ['RGIId', 'year', 'month'] + CLIMATE_VARS
    era = era_long[usecols].copy()
    era['year'] = pd.to_numeric(era['year'], errors='coerce').astype(int)
    era['month'] = pd.to_numeric(era['month'], errors='coerce').astype(int)
    era_lookup = {
        (row.RGIId, int(row.year), int(row.month)): {var: getattr(row, var) for var in CLIMATE_VARS}
        for row in era.itertuples(index=False)
    }

    full_grid = (
        era[['RGIId', 'year']]
        .drop_duplicates()
        .merge(meta_by_rgi, on='RGIId', how='inner')
        .sort_values(['RGIId', 'year'])
        .reset_index(drop=True)
    )
    full_grid['year'] = pd.to_numeric(full_grid['year'], errors='coerce').astype('Int64')
    full_grid['O1Region'] = normalize_region(full_grid['O1Region'])
    full_grid['O2Region'] = pd.to_numeric(full_grid['O2Region'], errors='coerce').astype('Int64')

    return mb, full_grid, era_lookup


def build_observed_window_frame(df, season):
    work = df.copy()
    if season == 'winter':
        start = work['begin_dt']
        end = work['mid_dt']
        target_col = 'winter_balance'
    else:
        start = work['mid_dt']
        end = work['end_dt']
        target_col = 'summer_balance'

    valid = start.notna() & end.notna() & (start <= end)
    shifted_start = shift_month(start.loc[valid], direction='start')
    shifted_end = shift_month(end.loc[valid], direction='end')
    keep = shifted_start <= shifted_end
    valid_index = start.loc[valid].index[keep]

    obs = work.loc[valid_index, [
        'RGIId', 'year', 'O1Region', 'O2Region', 'Area', 'CenLon', 'CenLat',
        'Zmin', 'Zmax', 'Zmed', 'Slope', 'Aspect', 'Lmax', 'Status', 'Connect',
        'Form', 'TermType', 'Surging', 'Linkages', 'Name', 'gtng_region',
        'glacier_name', 'glacier_id', 'time_system', target_col
    ]].copy()
    obs['season'] = season
    obs['mb'] = pd.to_numeric(obs[target_col], errors='coerce')
    shifted_start = shifted_start.loc[valid_index]
    shifted_end = shifted_end.loc[valid_index]
    obs['season_start_month'] = shifted_start.dt.month.to_numpy()
    obs['season_end_month'] = shifted_end.dt.month.to_numpy()
    obs['season_start_year_offset'] = (shifted_start.dt.year - work.loc[valid_index, 'year'].astype(int)).to_numpy()
    obs['season_end_year_offset'] = (shifted_end.dt.year - work.loc[valid_index, 'year'].astype(int)).to_numpy()
    obs['window_pair'] = obs.apply(lambda r: window_pair_label(window_key(r)), axis=1)
    obs['season_window_source'] = 'observed_month_window'
    obs['window_confidence'] = 'high'
    obs['window_vote_share'] = 1.0
    obs['neighbor_count'] = 0
    obs['neighbor_distance_median_km'] = np.nan
    obs['window_ensemble_size'] = 1
    return obs


def haversine_km(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 6371.0 * 2.0 * np.arcsin(np.sqrt(a))


def build_glacier_window_defaults(observed_all):
    rows = []
    for (rgiid, season), group in observed_all.groupby(['RGIId', 'season'], sort=False):
        key, share = dominant_window_pair(group)
        n = len(group)
        if n < 3 or key is None:
            continue
        rows.append({
            'RGIId': rgiid,
            'season': season,
            'n_obs_windows': int(n),
            'window_key': key,
            'window_pair': window_pair_label(key),
            'window_vote_share': share,
            'window_confidence': 'high' if n >= 5 else 'medium',
            'CenLon': float(pd.to_numeric(group['CenLon'], errors='coerce').median()),
            'CenLat': float(pd.to_numeric(group['CenLat'], errors='coerce').median()),
            'O1Region': group['O1Region'].dropna().iloc[0] if group['O1Region'].notna().any() else np.nan,
        })
    return pd.DataFrame(rows)


def build_region_window_distributions(observed_all):
    rows = []
    for (o1, season), group in observed_all.groupby(['O1Region', 'season'], sort=False):
        keys = group.apply(window_key, axis=1).dropna()
        if keys.empty:
            continue
        counts = keys.value_counts()
        total = counts.sum()
        for rank, (key, count) in enumerate(counts.head(REGION_ENSEMBLE_TOPK).items(), start=1):
            rows.append({
                'O1Region': o1,
                'season': season,
                'rank': rank,
                'window_key': key,
                'window_pair': window_pair_label(key),
                'window_share': float(count / total),
                'window_count': int(count),
                'total_windows': int(total),
            })
    return pd.DataFrame(rows)


def assign_nearest_neighbor_window(row, glacier_defaults, season):
    pool = glacier_defaults.loc[
        glacier_defaults['season'].eq(season)
        & glacier_defaults['O1Region'].eq(row['O1Region'])
        & glacier_defaults['CenLon'].notna()
        & glacier_defaults['CenLat'].notna()
    ].copy()
    pool = pool.loc[pool['RGIId'].ne(row['RGIId'])]
    if pool.empty or pd.isna(row['CenLon']) or pd.isna(row['CenLat']):
        return None
    pool['distance_km'] = haversine_km(float(row['CenLon']), float(row['CenLat']), pool['CenLon'].astype(float), pool['CenLat'].astype(float))
    nn = pool.nsmallest(K_NEIGHBORS, 'distance_km').copy()
    nn['weight'] = 1.0 / (nn['distance_km'] + 1.0)
    votes = nn.groupby('window_key')['weight'].sum().sort_values(ascending=False)
    top_key = votes.index[0]
    vote_share = float(votes.iloc[0] / votes.sum())
    return {
        'window_key': top_key,
        'window_vote_share': vote_share,
        'neighbor_count': int(len(nn)),
        'neighbor_distance_median_km': float(nn['distance_km'].median()),
    }


def build_reconstruction_candidates(mb, full_grid, season):
    target_col = 'winter_balance' if season == 'winter' else 'summer_balance'
    observed_w = build_observed_window_frame(mb, 'winter')
    observed_s = build_observed_window_frame(mb, 'summer')
    observed_all = pd.concat([observed_w, observed_s], ignore_index=True)
    glacier_defaults = build_glacier_window_defaults(observed_all)
    region_dist = build_region_window_distributions(observed_all)
    observed = observed_w if season == 'winter' else observed_s
    obs_cols = [
        'RGIId', 'year', 'mb', 'time_system',
        'season_start_month', 'season_end_month',
        'season_start_year_offset', 'season_end_year_offset',
        'window_pair', 'season_window_source', 'window_confidence',
        'window_vote_share', 'neighbor_count', 'neighbor_distance_median_km',
        'window_ensemble_size'
    ]

    base_cols = [
        'RGIId', 'year', 'O1Region', 'O2Region', 'Area', 'CenLon', 'CenLat',
        'Zmin', 'Zmax', 'Zmed', 'Slope', 'Aspect', 'Lmax', 'Status', 'Connect',
        'Form', 'TermType', 'Surging', 'Linkages', 'Name', 'gtng_region',
        'glacier_name', 'wgms_glacier_id'
    ]
    candidates = full_grid[base_cols].copy()
    candidates['season'] = season
    candidates = candidates.merge(observed[obs_cols], on=['RGIId', 'year'], how='left')
    candidates['mb'] = pd.to_numeric(candidates['mb'], errors='coerce')
    candidates['time_system'] = candidates['time_system'].fillna('reconstructed_window')
    candidates['glacier_id'] = candidates['wgms_glacier_id']

    normal_rows = []
    ensemble_rows = []
    gdef = glacier_defaults.loc[glacier_defaults['season'].eq(season)].set_index('RGIId', drop=False)
    rdist = region_dist.loc[region_dist['season'].eq(season)].copy()

    for _, row in candidates.iterrows():
        if pd.notna(row.get('season_start_month')):
            normal_rows.append(row.to_frame().T)
            continue

        if row['RGIId'] in gdef.index:
            gd = gdef.loc[row['RGIId']]
            key = gd['window_key']
            normal_rows.append(apply_window_key(
                row.to_frame().T, key, 'glacier_history', gd['window_confidence'],
                vote_share=gd['window_vote_share'], neighbor_count=0,
                neighbor_distance_median_km=np.nan, ensemble_size=1
            ))
            continue

        nn = assign_nearest_neighbor_window(row, glacier_defaults, season)
        if nn is not None and nn['window_vote_share'] >= NN_VOTE_THRESHOLD:
            normal_rows.append(apply_window_key(
                row.to_frame().T, nn['window_key'], 'nearest_neighbor', 'low',
                vote_share=nn['window_vote_share'], neighbor_count=nn['neighbor_count'],
                neighbor_distance_median_km=nn['neighbor_distance_median_km'], ensemble_size=1
            ))
            continue

        reg = rdist.loc[rdist['O1Region'].eq(row['O1Region'])].copy()
        if reg.empty:
            continue
        topk = reg.head(REGION_ENSEMBLE_TOPK)
        for _, rr in topk.iterrows():
            ensemble_rows.append(apply_window_key(
                row.to_frame().T, rr['window_key'], 'o1region_ensemble', 'very_low',
                vote_share=rr['window_share'], neighbor_count=0,
                neighbor_distance_median_km=np.nan, ensemble_size=int(len(topk))
            ))

    normal = pd.concat(normal_rows, ignore_index=True) if normal_rows else pd.DataFrame()
    ensemble = pd.concat(ensemble_rows, ignore_index=True) if ensemble_rows else pd.DataFrame()
    for df in [normal, ensemble]:
        if not df.empty:
            for col in ['season_start_month', 'season_end_month', 'season_start_year_offset', 'season_end_year_offset', 'year']:
                df[col] = pd.to_numeric(df[col], errors='coerce')
    return normal, ensemble


def classify_model_uncertainty(r2):
    if pd.isna(r2):
        return 'low'
    if r2 >= 0.75:
        return 'high'
    if r2 >= 0.5:
        return 'medium'
    return 'low'


def worst_uncertainty(model_unc, window_unc):
    rank = {'high': 0, 'medium': 1, 'low': 2, 'very_low': 3}
    inv = {0: 'high_confidence', 1: 'medium_confidence', 2: 'low_confidence', 3: 'low_confidence'}
    return inv[max(rank.get(model_unc, 2), rank.get(window_unc, 2))]



def add_region_name(df):
    df = df.copy()
    o1 = pd.to_numeric(df['O1Region'], errors='coerce').astype('Int64')
    df['region_name'] = o1.astype('string').radd('O1Region_')
    df.loc[o1.isna(), 'region_name'] = pd.NA
    return df

In [ ]:
DATA_PATH = OUT_DIR / 'wgms_global_winter_era5land_1979_2025.csv'
PAYLOAD_PATH = MODEL_DIR / 'preprocessed_winter_data.pkl'
PARAMS_PATH = MODEL_DIR / 'xgboost_winter_unweighted_params.pkl'
PRODUCT_PATH = RECON_DIR / 'winter_reconstruction_product_1979_2025.csv'
MODEL_PATH = RECON_DIR / 'xgboost_winter_reconstruction_model.pkl'
GLOBAL_MODEL_PATH = MODEL_DIR / 'xgboost_winter_global.pkl'

mb, full_grid, era_lookup = prepare_base_tables()
payload = dill.load(open(PAYLOAD_PATH, 'rb'))
params = dill.load(open(PARAMS_PATH, 'rb'))
feature_columns = list(payload['feature_columns'])

train_data = pd.read_csv(DATA_PATH, low_memory=False)
missing_payload_features = [c for c in feature_columns if c not in train_data.columns]
if missing_payload_features:
    print('Dropping stale payload features not present in monthly seasonal data:', missing_payload_features)
feature_columns = [c for c in feature_columns if c in train_data.columns]
for col in feature_columns:
    train_data[col] = pd.to_numeric(train_data[col], errors='coerce')
train_df = train_data.dropna(subset=['mb'] + feature_columns).copy().reset_index(drop=True)
if GLOBAL_MODEL_PATH.exists():
    model = dill.load(open(GLOBAL_MODEL_PATH, 'rb'))
    if model.get_booster().attr('sample_weighting') != 'none':
        raise RuntimeError('Stored winter global model is not the unweighted model. Rerun 03_Model_training.ipynb first.')
    print({'loaded_global_model': str(GLOBAL_MODEL_PATH), 'features': len(feature_columns)})
else:
    model = XGBRegressor(objective='reg:squarederror', n_jobs=-1, random_state=42, tree_method='hist', **params)
    model.fit(train_df[feature_columns], train_df['mb'].to_numpy(dtype=float))
    model.get_booster().set_attr(sample_weighting='none')
    print({'training_rows': len(train_df), 'features': len(feature_columns)})
dill.dump(model, open(MODEL_PATH, 'wb'))

In [ ]:
def predict_single_window(df, pred_col):
    if df.empty:
        return df
    climate = aggregate_month_window(df, era_lookup)
    out = pd.concat([df.reset_index(drop=True), climate], axis=1)
    for col in feature_columns:
        out[col] = pd.to_numeric(out[col], errors='coerce')
    out = out.dropna(subset=feature_columns).copy().reset_index(drop=True)
    if out.empty:
        return out
    out[pred_col] = model.predict(out[feature_columns])
    return out


normal_candidates, ensemble_candidates = build_reconstruction_candidates(mb, full_grid, 'winter')
normal_pred = predict_single_window(normal_candidates, 'winter_pred')
ensemble_pred_long = predict_single_window(ensemble_candidates, 'winter_pred_member')

if not ensemble_pred_long.empty:
    keys = ['RGIId', 'year']
    agg = ensemble_pred_long.groupby(keys).agg(
        winter_pred=('winter_pred_member', 'mean'),
        winter_pred_std=('winter_pred_member', 'std'),
        winter_pred_min=('winter_pred_member', 'min'),
        winter_pred_max=('winter_pred_member', 'max'),
        window_ensemble_size=('winter_pred_member', 'count'),
        window_vote_share=('window_vote_share', 'max'),
        season_start_month=('season_start_month', lambda s: s.iloc[0]),
        season_end_month=('season_end_month', lambda s: s.iloc[0]),
        season_start_year_offset=('season_start_year_offset', lambda s: s.iloc[0]),
        season_end_year_offset=('season_end_year_offset', lambda s: s.iloc[0]),
        window_pair=('window_pair', lambda s: '|'.join(s.astype(str).tolist())),
    ).reset_index()
    # Preserve static glacier metadata (especially Area) when ensemble members are collapsed.
    # These columns are model features, but they are also required by downstream regional aggregation.
    static_meta = [c for c in STATIC_FEATURES if c in ensemble_pred_long.columns]
    other_meta = [c for c in ensemble_pred_long.columns if c not in CLIMATE_VARS + feature_columns + ['winter_pred_member', 'winter_pred', 'winter_pred_std', 'winter_pred_min', 'winter_pred_max'] + keys]
    meta_cols = list(dict.fromkeys(keys + static_meta + other_meta))
    meta = ensemble_pred_long[meta_cols].drop_duplicates(subset=keys, keep='first').copy()
    meta = meta.drop(columns=[c for c in ['season_start_month', 'season_end_month', 'season_start_year_offset', 'season_end_year_offset', 'window_pair', 'window_vote_share', 'window_ensemble_size'] if c in meta.columns])
    ensemble_pred = meta.merge(agg, on=keys, how='inner')
    ensemble_pred['winter_pred_std'] = ensemble_pred['winter_pred_std'].fillna(0.0)
else:
    ensemble_pred = pd.DataFrame()

if not normal_pred.empty:
    normal_pred['winter_pred_std'] = 0.0
    normal_pred['winter_pred_min'] = normal_pred['winter_pred']
    normal_pred['winter_pred_max'] = normal_pred['winter_pred']

pred_df = pd.concat([normal_pred, ensemble_pred], ignore_index=True)
pred_df = add_region_name(pred_df)
pred_df['winter_global_pred'] = pred_df['winter_pred']
pred_df['winter_pred'] = pred_df['winter_global_pred']

pred_df['winter_obs'] = pred_df['mb']
pred_df['winter_recon'] = pred_df['winter_obs'].where(pred_df['winter_obs'].notna(), pred_df['winter_global_pred'])
pred_df['winter_source_flag'] = np.where(pred_df['winter_obs'].notna(), 'observed', 'gap_filled')
pred_df['model_type'] = 'winter_xgb_global'
pred_df['validation_regime_reference'] = 'loyo'

pred_df['model_uncertainty'] = 'high'
pred_df['uncertainty_class'] = [
    worst_uncertainty(m, w) for m, w in zip(pred_df['model_uncertainty'], pred_df['window_confidence'])
]
pred_df.loc[pred_df['season_window_source'].eq('o1region_ensemble'), 'uncertainty_class'] = 'low_confidence'

keep_cols = [
    'RGIId', 'year', 'O1Region', 'O2Region', 'region_name', 'Area',
    'glacier_name', 'gtng_region', 'time_system',
    'winter_obs', 'winter_global_pred',
    'winter_pred', 'winter_recon', 'winter_source_flag',
    'winter_pred_std', 'winter_pred_min', 'winter_pred_max',
    'season_window_source', 'window_confidence', 'window_pair', 'window_vote_share',
    'neighbor_count', 'neighbor_distance_median_km', 'window_ensemble_size',
    'model_type', 'validation_regime_reference',
    'uncertainty_class', 'season_start_month', 'season_end_month',
    'season_start_year_offset', 'season_end_year_offset'
]
pred_df[keep_cols].sort_values(['RGIId', 'year']).to_csv(PRODUCT_PATH, index=False)
print(PRODUCT_PATH)
print(pred_df['winter_source_flag'].value_counts(dropna=False).to_dict())
print(pred_df['season_window_source'].value_counts(dropna=False).to_dict())

## 2. Summer reconstruction

Summer reconstruction applies the same window-assignment framework used for winter reconstruction, but uses the summer model and season-specific analysis.

In [ ]:
from pathlib import Path
import json
import math
import dill
import numpy as np
import pandas as pd
from xgboost import XGBRegressor


WGMS_DIR = ROOT_DIR / "data/DOI-WGMS-FoG-2026-02-10/data"
TARGET_CSV = ROOT_DIR / "data/interim/wgms_targets/wgms_global_targets_with_rgi_1979_2025.csv"
ERA_LONG_CSV = ROOT_DIR / "data/raw/era5land_gee/wgms_global_era5land_1979_2025.csv"
OUT_DIR = ROOT_DIR / "data/processed/seasonal_training"
RECON_DIR = ROOT_DIR / "results/reconstruction"
MODEL_DIR = ROOT_DIR / "models/annual_seasonal"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RECON_DIR.mkdir(parents=True, exist_ok=True)

YEAR_MIN, YEAR_MAX = 1979, 2025
K_NEIGHBORS = 5
NN_VOTE_THRESHOLD = 0.5
REGION_ENSEMBLE_TOPK = 3
STATE_VARS = ['t2m', 'rsn', 'sde', 'tsn', 'fal', 'sp']
FLOW_VARS = ['sf', 'smlt', 'slhf', 'ssr', 'strd', 'sshf', 'ssrd', 'tp']
CLIMATE_VARS = STATE_VARS + FLOW_VARS


def parse_wgms_date(series):
    s = series.astype('string').str.strip()
    out = pd.to_datetime(pd.Series(pd.NaT, index=series.index))
    numeric_mask = s.str.fullmatch(r'\d{8}').fillna(False)
    out.loc[numeric_mask] = pd.to_datetime(s.loc[numeric_mask], format='%Y%m%d', errors='coerce')
    out.loc[~numeric_mask] = pd.to_datetime(s.loc[~numeric_mask], errors='coerce')
    return out


def shift_month(date_series, direction):
    adjusted = date_series.copy()
    if direction == 'start':
        adjusted = adjusted.where(adjusted.dt.day <= 15, adjusted + pd.offsets.MonthBegin(1))
    elif direction == 'end':
        adjusted = adjusted.where(adjusted.dt.day >= 15, adjusted - pd.offsets.MonthEnd(1))
    else:
        raise ValueError("direction must be 'start' or 'end'")
    return adjusted


def normalize_region(series):
    return pd.to_numeric(series, errors='coerce').astype('Int64')


def window_key(row):
    vals = [
        row['season_start_month'], row['season_end_month'],
        row['season_start_year_offset'], row['season_end_year_offset'],
    ]
    if any(pd.isna(v) for v in vals):
        return None
    return (int(vals[0]), int(vals[1]), int(vals[2]), int(vals[3]))


def window_pair_label(key):
    if key is None:
        return np.nan
    sm, em, so, eo = key
    return f"{so:+d}:{sm:02d}->{eo:+d}:{em:02d}"


def dominant_window_pair(group):
    keys = group.apply(window_key, axis=1).dropna()
    if keys.empty:
        return None, np.nan
    counts = keys.value_counts()
    top_count = counts.iloc[0]
    top_keys = sorted(counts[counts == top_count].index.tolist())
    key = top_keys[0]
    return key, float(top_count / counts.sum())


def apply_window_key(df, key, source, confidence, vote_share=np.nan, neighbor_count=0, neighbor_distance_median_km=np.nan, ensemble_size=1):
    sm, em, so, eo = key
    df = df.copy()
    df['season_start_month'] = sm
    df['season_end_month'] = em
    df['season_start_year_offset'] = so
    df['season_end_year_offset'] = eo
    df['window_pair'] = window_pair_label(key)
    df['season_window_source'] = source
    df['window_confidence'] = confidence
    df['window_vote_share'] = vote_share
    df['neighbor_count'] = neighbor_count
    df['neighbor_distance_median_km'] = neighbor_distance_median_km
    df['window_ensemble_size'] = ensemble_size
    return df


def month_sequence(start_month, start_offset, end_month, end_offset, balance_year):
    start_year = int(balance_year) + int(start_offset)
    end_year = int(balance_year) + int(end_offset)
    start_period = pd.Period(year=start_year, month=int(start_month), freq='M')
    end_period = pd.Period(year=end_year, month=int(end_month), freq='M')
    if end_period < start_period:
        return []
    periods = pd.period_range(start_period, end_period, freq='M')
    return [(int(p.year), int(p.month)) for p in periods]


def aggregate_month_window(records, era_lookup):
    rows = []
    for rec in records.itertuples(index=False):
        months = month_sequence(rec.season_start_month, rec.season_start_year_offset, rec.season_end_month, rec.season_end_year_offset, rec.year)
        values = []
        for year_month in months:
            key = (rec.RGIId, year_month[0], year_month[1])
            if key in era_lookup:
                values.append(era_lookup[key])
        if len(values) == 0:
            out = {var: np.nan for var in CLIMATE_VARS}
        else:
            frame = pd.DataFrame(values)
            out = {var: frame[var].mean() for var in STATE_VARS}
            out.update({var: frame[var].sum(min_count=1) for var in FLOW_VARS})
        rows.append(out)
    return pd.DataFrame(rows)


def prepare_base_tables():
    mass_balance = pd.read_csv(WGMS_DIR / 'mass_balance.csv', low_memory=False)
    target_meta = pd.read_csv(TARGET_CSV, low_memory=False)
    era_long = pd.read_csv(ERA_LONG_CSV, low_memory=False)

    target_meta['wgms_glacier_id'] = pd.to_numeric(target_meta['wgms_glacier_id'], errors='coerce').astype('Int64')
    target_meta['O1Region'] = normalize_region(target_meta['O1Region'])
    target_meta['O2Region'] = pd.to_numeric(target_meta['O2Region'], errors='coerce').astype('Int64')
    mapped_ids = target_meta['wgms_glacier_id'].dropna().unique().tolist()

    base_meta_cols = [
        'RGIId', 'GLIMSId', 'CenLon', 'CenLat', 'O1Region', 'O2Region', 'Area',
        'Zmin', 'Zmax', 'Zmed', 'Slope', 'Aspect', 'Lmax', 'Status', 'Connect',
        'Form', 'TermType', 'Surging', 'Linkages', 'Name', 'gtng_region',
        'glacier_name', 'wgms_glacier_id'
    ]
    meta_map = (
        target_meta[base_meta_cols]
        .dropna(subset=['wgms_glacier_id'])
        .sort_values(['RGIId'])
        .drop_duplicates(subset=['wgms_glacier_id'], keep='first')
        .reset_index(drop=True)
    )
    meta_by_rgi = (
        target_meta[base_meta_cols]
        .sort_values(['RGIId'])
        .drop_duplicates(subset=['RGIId'], keep='first')
        .reset_index(drop=True)
    )

    mb = mass_balance.copy()
    mb['glacier_id'] = pd.to_numeric(mb['glacier_id'], errors='coerce').astype('Int64')
    mb['year'] = pd.to_numeric(mb['year'], errors='coerce').astype('Int64')
    mb = mb.loc[mb['glacier_id'].isin(mapped_ids) & mb['year'].between(YEAR_MIN, YEAR_MAX)].copy()
    mb = mb.drop(columns=[c for c in ['glacier_name', 'gtng_region'] if c in mb.columns])
    mb = mb.merge(meta_map, left_on='glacier_id', right_on='wgms_glacier_id', how='inner')

    mb['begin_dt'] = parse_wgms_date(mb['begin_date'])
    mb['mid_dt'] = parse_wgms_date(mb['midseason_date'])
    mb['end_dt'] = parse_wgms_date(mb['end_date'])
    mb['O1Region'] = normalize_region(mb['O1Region'])
    mb['O2Region'] = pd.to_numeric(mb['O2Region'], errors='coerce').astype('Int64')

    usecols = ['RGIId', 'year', 'month'] + CLIMATE_VARS
    era = era_long[usecols].copy()
    era['year'] = pd.to_numeric(era['year'], errors='coerce').astype(int)
    era['month'] = pd.to_numeric(era['month'], errors='coerce').astype(int)
    era_lookup = {
        (row.RGIId, int(row.year), int(row.month)): {var: getattr(row, var) for var in CLIMATE_VARS}
        for row in era.itertuples(index=False)
    }

    full_grid = (
        era[['RGIId', 'year']]
        .drop_duplicates()
        .merge(meta_by_rgi, on='RGIId', how='inner')
        .sort_values(['RGIId', 'year'])
        .reset_index(drop=True)
    )
    full_grid['year'] = pd.to_numeric(full_grid['year'], errors='coerce').astype('Int64')
    full_grid['O1Region'] = normalize_region(full_grid['O1Region'])
    full_grid['O2Region'] = pd.to_numeric(full_grid['O2Region'], errors='coerce').astype('Int64')

    return mb, full_grid, era_lookup


def build_observed_window_frame(df, season):
    work = df.copy()
    if season == 'winter':
        start = work['begin_dt']
        end = work['mid_dt']
        target_col = 'winter_balance'
    else:
        start = work['mid_dt']
        end = work['end_dt']
        target_col = 'summer_balance'

    valid = start.notna() & end.notna() & (start <= end)
    shifted_start = shift_month(start.loc[valid], direction='start')
    shifted_end = shift_month(end.loc[valid], direction='end')
    keep = shifted_start <= shifted_end
    valid_index = start.loc[valid].index[keep]

    obs = work.loc[valid_index, [
        'RGIId', 'year', 'O1Region', 'O2Region', 'Area', 'CenLon', 'CenLat',
        'Zmin', 'Zmax', 'Zmed', 'Slope', 'Aspect', 'Lmax', 'Status', 'Connect',
        'Form', 'TermType', 'Surging', 'Linkages', 'Name', 'gtng_region',
        'glacier_name', 'glacier_id', 'time_system', target_col
    ]].copy()
    obs['season'] = season
    obs['mb'] = pd.to_numeric(obs[target_col], errors='coerce')
    shifted_start = shifted_start.loc[valid_index]
    shifted_end = shifted_end.loc[valid_index]
    obs['season_start_month'] = shifted_start.dt.month.to_numpy()
    obs['season_end_month'] = shifted_end.dt.month.to_numpy()
    obs['season_start_year_offset'] = (shifted_start.dt.year - work.loc[valid_index, 'year'].astype(int)).to_numpy()
    obs['season_end_year_offset'] = (shifted_end.dt.year - work.loc[valid_index, 'year'].astype(int)).to_numpy()
    obs['window_pair'] = obs.apply(lambda r: window_pair_label(window_key(r)), axis=1)
    obs['season_window_source'] = 'observed_month_window'
    obs['window_confidence'] = 'high'
    obs['window_vote_share'] = 1.0
    obs['neighbor_count'] = 0
    obs['neighbor_distance_median_km'] = np.nan
    obs['window_ensemble_size'] = 1
    return obs


def haversine_km(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 6371.0 * 2.0 * np.arcsin(np.sqrt(a))


def build_glacier_window_defaults(observed_all):
    rows = []
    for (rgiid, season), group in observed_all.groupby(['RGIId', 'season'], sort=False):
        key, share = dominant_window_pair(group)
        n = len(group)
        if n < 3 or key is None:
            continue
        rows.append({
            'RGIId': rgiid,
            'season': season,
            'n_obs_windows': int(n),
            'window_key': key,
            'window_pair': window_pair_label(key),
            'window_vote_share': share,
            'window_confidence': 'high' if n >= 5 else 'medium',
            'CenLon': float(pd.to_numeric(group['CenLon'], errors='coerce').median()),
            'CenLat': float(pd.to_numeric(group['CenLat'], errors='coerce').median()),
            'O1Region': group['O1Region'].dropna().iloc[0] if group['O1Region'].notna().any() else np.nan,
        })
    return pd.DataFrame(rows)


def build_region_window_distributions(observed_all):
    rows = []
    for (o1, season), group in observed_all.groupby(['O1Region', 'season'], sort=False):
        keys = group.apply(window_key, axis=1).dropna()
        if keys.empty:
            continue
        counts = keys.value_counts()
        total = counts.sum()
        for rank, (key, count) in enumerate(counts.head(REGION_ENSEMBLE_TOPK).items(), start=1):
            rows.append({
                'O1Region': o1,
                'season': season,
                'rank': rank,
                'window_key': key,
                'window_pair': window_pair_label(key),
                'window_share': float(count / total),
                'window_count': int(count),
                'total_windows': int(total),
            })
    return pd.DataFrame(rows)


def assign_nearest_neighbor_window(row, glacier_defaults, season):
    pool = glacier_defaults.loc[
        glacier_defaults['season'].eq(season)
        & glacier_defaults['O1Region'].eq(row['O1Region'])
        & glacier_defaults['CenLon'].notna()
        & glacier_defaults['CenLat'].notna()
    ].copy()
    pool = pool.loc[pool['RGIId'].ne(row['RGIId'])]
    if pool.empty or pd.isna(row['CenLon']) or pd.isna(row['CenLat']):
        return None
    pool['distance_km'] = haversine_km(float(row['CenLon']), float(row['CenLat']), pool['CenLon'].astype(float), pool['CenLat'].astype(float))
    nn = pool.nsmallest(K_NEIGHBORS, 'distance_km').copy()
    nn['weight'] = 1.0 / (nn['distance_km'] + 1.0)
    votes = nn.groupby('window_key')['weight'].sum().sort_values(ascending=False)
    top_key = votes.index[0]
    vote_share = float(votes.iloc[0] / votes.sum())
    return {
        'window_key': top_key,
        'window_vote_share': vote_share,
        'neighbor_count': int(len(nn)),
        'neighbor_distance_median_km': float(nn['distance_km'].median()),
    }


def build_reconstruction_candidates(mb, full_grid, season):
    target_col = 'winter_balance' if season == 'winter' else 'summer_balance'
    observed_w = build_observed_window_frame(mb, 'winter')
    observed_s = build_observed_window_frame(mb, 'summer')
    observed_all = pd.concat([observed_w, observed_s], ignore_index=True)
    glacier_defaults = build_glacier_window_defaults(observed_all)
    region_dist = build_region_window_distributions(observed_all)
    observed = observed_w if season == 'winter' else observed_s
    obs_cols = [
        'RGIId', 'year', 'mb', 'time_system',
        'season_start_month', 'season_end_month',
        'season_start_year_offset', 'season_end_year_offset',
        'window_pair', 'season_window_source', 'window_confidence',
        'window_vote_share', 'neighbor_count', 'neighbor_distance_median_km',
        'window_ensemble_size'
    ]

    base_cols = [
        'RGIId', 'year', 'O1Region', 'O2Region', 'Area', 'CenLon', 'CenLat',
        'Zmin', 'Zmax', 'Zmed', 'Slope', 'Aspect', 'Lmax', 'Status', 'Connect',
        'Form', 'TermType', 'Surging', 'Linkages', 'Name', 'gtng_region',
        'glacier_name', 'wgms_glacier_id'
    ]
    candidates = full_grid[base_cols].copy()
    candidates['season'] = season
    candidates = candidates.merge(observed[obs_cols], on=['RGIId', 'year'], how='left')
    candidates['mb'] = pd.to_numeric(candidates['mb'], errors='coerce')
    candidates['time_system'] = candidates['time_system'].fillna('reconstructed_window')
    candidates['glacier_id'] = candidates['wgms_glacier_id']

    normal_rows = []
    ensemble_rows = []
    gdef = glacier_defaults.loc[glacier_defaults['season'].eq(season)].set_index('RGIId', drop=False)
    rdist = region_dist.loc[region_dist['season'].eq(season)].copy()

    for _, row in candidates.iterrows():
        if pd.notna(row.get('season_start_month')):
            normal_rows.append(row.to_frame().T)
            continue

        if row['RGIId'] in gdef.index:
            gd = gdef.loc[row['RGIId']]
            key = gd['window_key']
            normal_rows.append(apply_window_key(
                row.to_frame().T, key, 'glacier_history', gd['window_confidence'],
                vote_share=gd['window_vote_share'], neighbor_count=0,
                neighbor_distance_median_km=np.nan, ensemble_size=1
            ))
            continue

        nn = assign_nearest_neighbor_window(row, glacier_defaults, season)
        if nn is not None and nn['window_vote_share'] >= NN_VOTE_THRESHOLD:
            normal_rows.append(apply_window_key(
                row.to_frame().T, nn['window_key'], 'nearest_neighbor', 'low',
                vote_share=nn['window_vote_share'], neighbor_count=nn['neighbor_count'],
                neighbor_distance_median_km=nn['neighbor_distance_median_km'], ensemble_size=1
            ))
            continue

        reg = rdist.loc[rdist['O1Region'].eq(row['O1Region'])].copy()
        if reg.empty:
            continue
        topk = reg.head(REGION_ENSEMBLE_TOPK)
        for _, rr in topk.iterrows():
            ensemble_rows.append(apply_window_key(
                row.to_frame().T, rr['window_key'], 'o1region_ensemble', 'very_low',
                vote_share=rr['window_share'], neighbor_count=0,
                neighbor_distance_median_km=np.nan, ensemble_size=int(len(topk))
            ))

    normal = pd.concat(normal_rows, ignore_index=True) if normal_rows else pd.DataFrame()
    ensemble = pd.concat(ensemble_rows, ignore_index=True) if ensemble_rows else pd.DataFrame()
    for df in [normal, ensemble]:
        if not df.empty:
            for col in ['season_start_month', 'season_end_month', 'season_start_year_offset', 'season_end_year_offset', 'year']:
                df[col] = pd.to_numeric(df[col], errors='coerce')
    return normal, ensemble


def classify_model_uncertainty(r2):
    if pd.isna(r2):
        return 'low'
    if r2 >= 0.75:
        return 'high'
    if r2 >= 0.5:
        return 'medium'
    return 'low'


def worst_uncertainty(model_unc, window_unc):
    rank = {'high': 0, 'medium': 1, 'low': 2, 'very_low': 3}
    inv = {0: 'high_confidence', 1: 'medium_confidence', 2: 'low_confidence', 3: 'low_confidence'}
    return inv[max(rank.get(model_unc, 2), rank.get(window_unc, 2))]



def add_region_name(df):
    df = df.copy()
    o1 = pd.to_numeric(df['O1Region'], errors='coerce').astype('Int64')
    df['region_name'] = o1.astype('string').radd('O1Region_')
    df.loc[o1.isna(), 'region_name'] = pd.NA
    return df

In [ ]:
DATA_PATH = OUT_DIR / 'wgms_global_summer_era5land_1979_2025.csv'
PAYLOAD_PATH = MODEL_DIR / 'preprocessed_summer_data.pkl'
PARAMS_PATH = MODEL_DIR / 'xgboost_summer_unweighted_params.pkl'
PRODUCT_PATH = RECON_DIR / 'summer_reconstruction_product_1979_2025.csv'
MODEL_PATH = RECON_DIR / 'xgboost_summer_reconstruction_model.pkl'
GLOBAL_MODEL_PATH = MODEL_DIR / 'xgboost_summer_global.pkl'

mb, full_grid, era_lookup = prepare_base_tables()
payload = dill.load(open(PAYLOAD_PATH, 'rb'))
params = dill.load(open(PARAMS_PATH, 'rb'))
feature_columns = list(payload['feature_columns'])

train_data = pd.read_csv(DATA_PATH, low_memory=False)
missing_payload_features = [c for c in feature_columns if c not in train_data.columns]
if missing_payload_features:
    print('Dropping stale payload features not present in monthly seasonal data:', missing_payload_features)
feature_columns = [c for c in feature_columns if c in train_data.columns]
for col in feature_columns:
    train_data[col] = pd.to_numeric(train_data[col], errors='coerce')
train_df = train_data.dropna(subset=['mb'] + feature_columns).copy().reset_index(drop=True)
if GLOBAL_MODEL_PATH.exists():
    model = dill.load(open(GLOBAL_MODEL_PATH, 'rb'))
    if model.get_booster().attr('sample_weighting') != 'none':
        raise RuntimeError('Stored summer global model is not the unweighted model. Rerun 03_Model_training.ipynb first.')
    print({'loaded_global_model': str(GLOBAL_MODEL_PATH), 'features': len(feature_columns)})
else:
    model = XGBRegressor(objective='reg:squarederror', n_jobs=-1, random_state=42, tree_method='hist', **params)
    model.fit(train_df[feature_columns], train_df['mb'].to_numpy(dtype=float))
    model.get_booster().set_attr(sample_weighting='none')
    print({'training_rows': len(train_df), 'features': len(feature_columns)})
dill.dump(model, open(MODEL_PATH, 'wb'))

In [ ]:
def predict_single_window(df, pred_col):
    if df.empty:
        return df
    climate = aggregate_month_window(df, era_lookup)
    out = pd.concat([df.reset_index(drop=True), climate], axis=1)
    for col in feature_columns:
        out[col] = pd.to_numeric(out[col], errors='coerce')
    out = out.dropna(subset=feature_columns).copy().reset_index(drop=True)
    if out.empty:
        return out
    out[pred_col] = model.predict(out[feature_columns])
    return out


normal_candidates, ensemble_candidates = build_reconstruction_candidates(mb, full_grid, 'summer')
normal_pred = predict_single_window(normal_candidates, 'summer_pred')
ensemble_pred_long = predict_single_window(ensemble_candidates, 'summer_pred_member')

if not ensemble_pred_long.empty:
    keys = ['RGIId', 'year']
    agg = ensemble_pred_long.groupby(keys).agg(
        summer_pred=('summer_pred_member', 'mean'),
        summer_pred_std=('summer_pred_member', 'std'),
        summer_pred_min=('summer_pred_member', 'min'),
        summer_pred_max=('summer_pred_member', 'max'),
        window_ensemble_size=('summer_pred_member', 'count'),
        window_vote_share=('window_vote_share', 'max'),
        season_start_month=('season_start_month', lambda s: s.iloc[0]),
        season_end_month=('season_end_month', lambda s: s.iloc[0]),
        season_start_year_offset=('season_start_year_offset', lambda s: s.iloc[0]),
        season_end_year_offset=('season_end_year_offset', lambda s: s.iloc[0]),
        window_pair=('window_pair', lambda s: '|'.join(s.astype(str).tolist())),
    ).reset_index()
    # Preserve static glacier metadata (especially Area) when ensemble members are collapsed.
    # These columns are model features, but they are also required by downstream regional aggregation.
    static_meta = [c for c in STATIC_FEATURES if c in ensemble_pred_long.columns]
    other_meta = [c for c in ensemble_pred_long.columns if c not in CLIMATE_VARS + feature_columns + ['summer_pred_member', 'summer_pred', 'summer_pred_std', 'summer_pred_min', 'summer_pred_max'] + keys]
    meta_cols = list(dict.fromkeys(keys + static_meta + other_meta))
    meta = ensemble_pred_long[meta_cols].drop_duplicates(subset=keys, keep='first').copy()
    meta = meta.drop(columns=[c for c in ['season_start_month', 'season_end_month', 'season_start_year_offset', 'season_end_year_offset', 'window_pair', 'window_vote_share', 'window_ensemble_size'] if c in meta.columns])
    ensemble_pred = meta.merge(agg, on=keys, how='inner')
    ensemble_pred['summer_pred_std'] = ensemble_pred['summer_pred_std'].fillna(0.0)
else:
    ensemble_pred = pd.DataFrame()

if not normal_pred.empty:
    normal_pred['summer_pred_std'] = 0.0
    normal_pred['summer_pred_min'] = normal_pred['summer_pred']
    normal_pred['summer_pred_max'] = normal_pred['summer_pred']

pred_df = pd.concat([normal_pred, ensemble_pred], ignore_index=True)
pred_df = add_region_name(pred_df)
pred_df['summer_global_pred'] = pred_df['summer_pred']
pred_df['summer_pred'] = pred_df['summer_global_pred']

pred_df['summer_obs'] = pred_df['mb']
pred_df['summer_recon'] = pred_df['summer_obs'].where(pred_df['summer_obs'].notna(), pred_df['summer_global_pred'])
pred_df['summer_source_flag'] = np.where(pred_df['summer_obs'].notna(), 'observed', 'gap_filled')
pred_df['model_type'] = 'summer_xgb_global'
pred_df['validation_regime_reference'] = 'loyo'

pred_df['model_uncertainty'] = 'high'
pred_df['uncertainty_class'] = [
    worst_uncertainty(m, w) for m, w in zip(pred_df['model_uncertainty'], pred_df['window_confidence'])
]
pred_df.loc[pred_df['season_window_source'].eq('o1region_ensemble'), 'uncertainty_class'] = 'low_confidence'

keep_cols = [
    'RGIId', 'year', 'O1Region', 'O2Region', 'region_name', 'Area',
    'glacier_name', 'gtng_region', 'time_system',
    'summer_obs', 'summer_global_pred',
    'summer_pred', 'summer_recon', 'summer_source_flag',
    'summer_pred_std', 'summer_pred_min', 'summer_pred_max',
    'season_window_source', 'window_confidence', 'window_pair', 'window_vote_share',
    'neighbor_count', 'neighbor_distance_median_km', 'window_ensemble_size',
    'model_type', 'validation_regime_reference',
    'uncertainty_class', 'season_start_month', 'season_end_month',
    'season_start_year_offset', 'season_end_year_offset'
]
pred_df[keep_cols].sort_values(['RGIId', 'year']).to_csv(PRODUCT_PATH, index=False)
print(PRODUCT_PATH)
print(pred_df['summer_source_flag'].value_counts(dropna=False).to_dict())
print(pred_df['season_window_source'].value_counts(dropna=False).to_dict())

## 3. Annual reconstruction

Annual reconstruction combines observed annual balances, seasonal-sum reconstruction where winter and summer predictions are both available, and transparent annual-model fallback where seasonal reconstruction is unsupported.

In [ ]:
from pathlib import Path
import json
import dill
import numpy as np
import pandas as pd
from xgboost import XGBRegressor


OUT_DIR = ROOT_DIR / "results/reconstruction"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ANNUAL_MODEL_DIR = ROOT_DIR / "models/annual"


def classify_model_uncertainty(r2):
    if pd.isna(r2):
        return 'low'
    if r2 >= 0.75:
        return 'high'
    if r2 >= 0.5:
        return 'medium'
    return 'low'


def worst_uncertainty(model_unc, window_unc):
    rank = {'high': 0, 'medium': 1, 'low': 2, 'very_low': 3}
    inv = {0: 'high_confidence', 1: 'medium_confidence', 2: 'low_confidence', 3: 'low_confidence'}
    return inv[max(rank.get(model_unc, 2), rank.get(window_unc, 2))]



def add_region_name(df):
    df = df.copy()
    o1 = pd.to_numeric(df['O1Region'], errors='coerce').astype('Int64')
    df['region_name'] = o1.astype('string').radd('O1Region_')
    df.loc[o1.isna(), 'region_name'] = pd.NA
    return df

In [ ]:
TARGET_CSV = ROOT_DIR / 'data/interim/wgms_targets/wgms_global_targets_with_rgi_1979_2025.csv'
ERA_LONG_CSV = ROOT_DIR / 'data/raw/era5land_gee/wgms_global_era5land_1979_2025.csv'
PAYLOAD_PATH = ANNUAL_MODEL_DIR / 'preprocessed_data.pkl'
PARAMS_PATH = ANNUAL_MODEL_DIR / 'xgboost_annual_unweighted_params.pkl'
WINTER_PRODUCT_PATH = OUT_DIR / 'winter_reconstruction_product_1979_2025.csv'
SUMMER_PRODUCT_PATH = OUT_DIR / 'summer_reconstruction_product_1979_2025.csv'
PRODUCT_PATH = OUT_DIR / 'annual_reconstruction_product_1979_2025.csv'
MODEL_PATH = OUT_DIR / 'xgboost_annual_reconstruction_model.pkl'
GLOBAL_MODEL_PATH = ANNUAL_MODEL_DIR / 'xgboost_annual_global.pkl'

target = pd.read_csv(TARGET_CSV, low_memory=False)
era = pd.read_csv(ERA_LONG_CSV, low_memory=False)
payload = dill.load(open(PAYLOAD_PATH, 'rb'))
params = dill.load(open(PARAMS_PATH, 'rb'))
feature_columns = list(payload['feature_columns'])

In [ ]:
target['O1Region'] = pd.to_numeric(target['O1Region'], errors='coerce').astype('Int64')
target['O2Region'] = pd.to_numeric(target['O2Region'], errors='coerce').astype('Int64')
meta_cols = ['RGIId', 'GLIMSId', 'CenLon', 'CenLat', 'O1Region', 'O2Region', 'Area', 'Zmin', 'Zmax', 'Zmed', 'Slope', 'Aspect', 'Lmax', 'Status', 'Connect', 'Form', 'TermType', 'Surging', 'Linkages', 'Name', 'gtng_region', 'glacier_name']
meta = target[meta_cols].drop_duplicates(subset=['RGIId']).copy()
annual_obs = target[['RGIId', 'year', 'mwe', 'time_system']].rename(columns={'mwe': 'annual_obs'}).copy()

CLIMATE_VARS = ['t2m', 'rsn', 'sde', 'sf', 'smlt', 'tsn', 'fal', 'slhf', 'ssr', 'strd', 'sshf', 'ssrd', 'sp', 'tp']
annual_wide = era[['RGIId', 'year', 'month'] + CLIMATE_VARS].pivot_table(index=['RGIId', 'year'], columns='month', values=CLIMATE_VARS)
annual_wide.columns = [f'{int(month)}_{var}' for var, month in annual_wide.columns]
annual_wide = annual_wide.reset_index()

annual_candidates = annual_wide.merge(meta, on='RGIId', how='inner').merge(annual_obs, on=['RGIId', 'year'], how='left')
annual_candidates['O1Region'] = pd.to_numeric(annual_candidates['O1Region'], errors='coerce').astype('Int64')
annual_candidates['O2Region'] = pd.to_numeric(annual_candidates['O2Region'], errors='coerce').astype('Int64')
annual_candidates['annual_source_flag'] = np.where(annual_candidates['annual_obs'].notna(), 'observed', 'gap_filled')

train_df = annual_candidates.dropna(subset=['annual_obs'] + feature_columns).copy().reset_index(drop=True)
if GLOBAL_MODEL_PATH.exists():
    annual_bundle = dill.load(open(GLOBAL_MODEL_PATH, 'rb'))
    if annual_bundle.get('sample_weighting') != 'none':
        raise RuntimeError('Stored annual global model is not the unweighted model. Rerun 03_Model_training.ipynb first.')
    model = annual_bundle['model']
    scaler = annual_bundle.get('scaler')
    print({'loaded_global_model': str(GLOBAL_MODEL_PATH), 'features': len(feature_columns)})
else:
    scaler = StandardScaler()
    model = XGBRegressor(objective='reg:squarederror', n_jobs=-1, random_state=42, tree_method='hist', **params)
    X_train = pd.DataFrame(scaler.fit_transform(train_df[feature_columns]), columns=feature_columns, index=train_df.index)
    model.fit(X_train, train_df['annual_obs'].to_numpy(dtype=float))
    print({'annual_training_rows': len(train_df), 'features': len(feature_columns)})
dill.dump({'model': model, 'scaler': scaler, 'feature_columns': feature_columns}, open(MODEL_PATH, 'wb'))


def predict_annual_global(df):
    X = df[feature_columns]
    if scaler is not None:
        X = pd.DataFrame(scaler.transform(X), columns=feature_columns, index=df.index)
    return model.predict(X)

In [ ]:
pred_df = annual_candidates.dropna(subset=feature_columns).copy().reset_index(drop=True)
pred_df['annual_model_global_pred'] = predict_annual_global(pred_df)

pred_df = add_region_name(pred_df)
pred_df['annual_model_pred'] = pred_df['annual_model_global_pred']

winter = pd.read_csv(WINTER_PRODUCT_PATH)
summer = pd.read_csv(SUMMER_PRODUCT_PATH)
wcols = [
    'RGIId', 'year', 'winter_global_pred',
    'winter_pred', 'winter_pred_std', 'winter_pred_min', 'winter_pred_max', 'winter_source_flag',
    'season_window_source', 'window_confidence', 'uncertainty_class'
]
scols = [
    'RGIId', 'year', 'summer_global_pred',
    'summer_pred', 'summer_pred_std', 'summer_pred_min', 'summer_pred_max', 'summer_source_flag',
    'season_window_source', 'window_confidence', 'uncertainty_class'
]
ws = winter[wcols].merge(summer[scols], on=['RGIId', 'year'], how='outer', suffixes=('_winter', '_summer'))
pred_df = pred_df.merge(ws, on=['RGIId', 'year'], how='left')

pred_df['annual_from_seasonal_global_pred'] = pred_df['winter_global_pred'] + pred_df['summer_global_pred']
pred_df['annual_from_seasonal_pred'] = pred_df['annual_from_seasonal_global_pred']
pred_df['annual_from_seasonal_pred_std'] = np.sqrt(pred_df['winter_pred_std'].fillna(0.0) ** 2 + pred_df['summer_pred_std'].fillna(0.0) ** 2)
pred_df.loc[pred_df['annual_from_seasonal_pred'].isna(), 'annual_from_seasonal_pred_std'] = np.nan
pred_df['annual_from_seasonal_pred_min'] = pred_df['winter_pred_min'] + pred_df['summer_pred_min']
pred_df['annual_from_seasonal_pred_max'] = pred_df['winter_pred_max'] + pred_df['summer_pred_max']

has_seasonal = pred_df['annual_from_seasonal_pred'].notna()
pred_df['seasonal_prediction_available'] = has_seasonal
pred_df['annual_consensus_global_pred'] = np.where(has_seasonal, pred_df['annual_from_seasonal_global_pred'], pred_df['annual_model_global_pred'])
pred_df['annual_consensus_pred'] = np.where(has_seasonal, pred_df['annual_from_seasonal_pred'], pred_df['annual_model_global_pred'])
pred_df['annual_method_flag'] = np.select(
    [pred_df['annual_obs'].notna(), has_seasonal],
    ['observed', 'seasonal_sum'],
    default='annual_model_fallback'
)
missing_winter = pred_df['winter_global_pred'].isna()
missing_summer = pred_df['summer_global_pred'].isna()
pred_df['annual_model_fallback_reason'] = np.select(
    [
        pred_df['annual_method_flag'].ne('annual_model_fallback'),
        missing_winter & missing_summer,
        missing_winter,
        missing_summer,
    ],
    [
        'not_fallback',
        'missing_winter_and_summer_seasonal',
        'missing_winter_seasonal',
        'missing_summer_seasonal',
    ],
    default='unknown'
)
pred_df['annual_recon'] = pred_df['annual_obs'].where(pred_df['annual_obs'].notna(), pred_df['annual_consensus_pred'])
pred_df['model_type'] = np.select(
    [pred_df['annual_method_flag'].eq('seasonal_sum'), pred_df['annual_method_flag'].eq('annual_model_fallback')],
    ['annual_from_seasonal', 'annual_xgb'],
    default='observed'
)
pred_df['validation_regime_reference'] = 'loyo'

pred_df['model_uncertainty'] = 'medium'
map_to_short = {'high_confidence': 'high', 'medium_confidence': 'medium', 'low_confidence': 'low'}
seasonal_unc = []
for a, b in pred_df[['uncertainty_class_winter', 'uncertainty_class_summer']].fillna('low_confidence').to_numpy():
    vals = [map_to_short.get(a, 'low'), map_to_short.get(b, 'low')]
    seasonal_unc.append('high' if vals == ['high', 'high'] else ('medium' if 'low' not in vals else 'low'))
pred_df['seasonal_uncertainty'] = seasonal_unc
pred_df['uncertainty_class'] = np.where(
    pred_df['annual_method_flag'].eq('seasonal_sum'),
    [worst_uncertainty('high', s) for s in pred_df['seasonal_uncertainty']],
    [worst_uncertainty(m, 'high') for m in pred_df['model_uncertainty']]
)


keep_cols = [
    'RGIId', 'year', 'O1Region', 'O2Region', 'region_name', 'Area',
    'glacier_name', 'gtng_region', 'time_system',
    'annual_obs', 'annual_model_global_pred', 'annual_model_pred',
    'annual_from_seasonal_global_pred', 'annual_from_seasonal_pred',
    'annual_from_seasonal_pred_std', 'annual_from_seasonal_pred_min', 'annual_from_seasonal_pred_max',
    'annual_consensus_global_pred', 'annual_consensus_pred', 'annual_recon', 'annual_source_flag',
    'annual_method_flag', 'annual_model_fallback_reason', 'seasonal_prediction_available',
    'model_type', 'validation_regime_reference', 'uncertainty_class',
    'winter_global_pred', 'summer_global_pred',
    'winter_pred', 'summer_pred', 'winter_pred_std', 'summer_pred_std',
    'winter_source_flag', 'summer_source_flag',
    'season_window_source_winter', 'season_window_source_summer',
    'window_confidence_winter', 'window_confidence_summer'
]
pred_df[keep_cols].sort_values(['RGIId', 'year']).to_csv(PRODUCT_PATH, index=False)
print(PRODUCT_PATH)
print(pred_df['annual_method_flag'].value_counts(dropna=False).to_dict())

## 4. Product summary and regional analysis

Summarize regional glacier-year balances, seasonal-window coverage and annual model fallback reasons.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print


OUT_DIR = ROOT_DIR / "results/reconstruction"
WINTER_PRODUCT = OUT_DIR / 'winter_reconstruction_product_1979_2025.csv'
SUMMER_PRODUCT = OUT_DIR / 'summer_reconstruction_product_1979_2025.csv'
ANNUAL_PRODUCT = OUT_DIR / 'annual_reconstruction_product_1979_2025.csv'

REGION_WINTER = OUT_DIR / 'regional_winter_balance_summary.csv'
REGION_SUMMER = OUT_DIR / 'regional_summer_balance_summary.csv'
REGION_ANNUAL = OUT_DIR / 'regional_annual_balance_summary.csv'
COVERAGE_SUMMARY = OUT_DIR / 'seasonal_reconstruction_coverage_summary.csv'
ANNUAL_FALLBACK_REASON_SUMMARY = OUT_DIR / 'annual_model_fallback_reason_summary.csv'

In [ ]:
winter = pd.read_csv(WINTER_PRODUCT)
summer = pd.read_csv(SUMMER_PRODUCT)
annual = pd.read_csv(ANNUAL_PRODUCT)

for df in [winter, summer, annual]:
    df['O1Region'] = pd.to_numeric(df['O1Region'], errors='coerce').astype('Int64')
    df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')

In [ ]:
def area_weighted_mean(values, weights):
    valid = values.notna() & weights.notna()
    if valid.sum() == 0:
        return np.nan
    return float(np.average(values.loc[valid], weights=weights.loc[valid]))


def summarize_product(df, obs_col, recon_col, source_col, uncertainty_col=None):
    rows = []
    for (region, year), group in df.groupby(['O1Region', 'year'], sort=True):
        row = {
            'O1Region': region,
            'region_name': group['region_name'].dropna().iloc[0] if 'region_name' in group and group['region_name'].notna().any() else np.nan,
            'year': year,
            'n_glaciers_total': int(len(group)),
            'n_glaciers_observed': int((group[source_col] == 'observed').sum()),
            'n_glaciers_gap_filled': int((group[source_col] == 'gap_filled').sum()),
            'mean_pred': float(group[recon_col].mean()),
            'median_pred': float(group[recon_col].median()),
            'area_weighted_mean_pred': area_weighted_mean(group[recon_col], group['Area']),
            'mean_obs': float(group[obs_col].mean()) if group[obs_col].notna().any() else np.nan,
            'area_weighted_mean_obs': area_weighted_mean(group[obs_col], group['Area']),
        }
        if uncertainty_col and uncertainty_col in group.columns:
            row['mean_window_uncertainty'] = float(group[uncertainty_col].mean()) if group[uncertainty_col].notna().any() else np.nan
            row['median_window_uncertainty'] = float(group[uncertainty_col].median()) if group[uncertainty_col].notna().any() else np.nan
        if 'season_window_source' in group.columns:
            row['n_o1region_ensemble'] = int(group['season_window_source'].eq('o1region_ensemble').sum())
            row['share_o1region_ensemble'] = row['n_o1region_ensemble'] / row['n_glaciers_total'] if row['n_glaciers_total'] else np.nan
        elif {'season_window_source_winter', 'season_window_source_summer'}.issubset(group.columns):
            ens = group['season_window_source_winter'].eq('o1region_ensemble') | group['season_window_source_summer'].eq('o1region_ensemble')
            row['n_o1region_ensemble'] = int(ens.sum())
            row['share_o1region_ensemble'] = row['n_o1region_ensemble'] / row['n_glaciers_total'] if row['n_glaciers_total'] else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


regional_winter = summarize_product(winter, 'winter_obs', 'winter_recon', 'winter_source_flag', 'winter_pred_std')
regional_summer = summarize_product(summer, 'summer_obs', 'summer_recon', 'summer_source_flag', 'summer_pred_std')
regional_annual = summarize_product(annual, 'annual_obs', 'annual_recon', 'annual_source_flag', 'annual_from_seasonal_pred_std')
regional_winter.to_csv(REGION_WINTER, index=False)
regional_summer.to_csv(REGION_SUMMER, index=False)
regional_annual.to_csv(REGION_ANNUAL, index=False)
display(regional_annual.head())

In [ ]:
coverage_rows = []
for season_name, df in [('winter', winter), ('summer', summer)]:
    for source, group in df.groupby('season_window_source', dropna=False):
        coverage_rows.append({
            'season': season_name,
            'coverage_category': str(source),
            'count': int(len(group)),
            'observed_targets': int((group[f'{season_name}_source_flag'] == 'observed').sum()),
            'gap_filled_targets': int((group[f'{season_name}_source_flag'] == 'gap_filled').sum()),
        })
coverage = pd.DataFrame(coverage_rows).sort_values(['season', 'coverage_category'])
coverage.to_csv(COVERAGE_SUMMARY, index=False)

fallback_reason = (
    annual.loc[annual['annual_method_flag'].eq('annual_model_fallback')]
    .groupby(['O1Region', 'region_name', 'annual_model_fallback_reason'], dropna=False)
    .agg(n_records=('RGIId', 'size'), n_glaciers=('RGIId', 'nunique'), n_years=('year', 'nunique'))
    .reset_index()
    .sort_values(['n_records'], ascending=False)
)
fallback_reason.to_csv(ANNUAL_FALLBACK_REASON_SUMMARY, index=False)
display(fallback_reason.head(20))
coverage